# 10주차: 결측치 및 범주형 데이터 처리


## 1. 누락 데이터


### 1-1. 누락 데이터 개요

- 누락 데이터(Not a Number, NaN)는 일반적으로 유효한 데이터 값이 존재하지 않는 경우를 의미함
  - 실제 수집된 데이터에는 입력 오류나 측정 실패로 인해 누락 데이터가 포함되는 경우가 많음
  - 누락 데이터가 존재하면 연산 함수가 정상적으로 적용되지 않거나 분석 결과가 왜곡될 수 있음
  - 실제 데이터 분석에서는 누락 데이터 여부를 먼저 확인하고 제거 또는 대체 과정을 거친 후 분석을 수행해야 함


### 1-2. 누락 데이터 확인

- `info()` 함수는 데이터프레임의 요약 정보를 출력함
  - 각 열의 데이터 타입과 전체 행 중에서 유효한 값(Non-Null) 개수를 보여줌
  - 이를 통해 누락 데이터가 존재하는 열을 빠르게 파악할 수 있음
  - 누락 데이터가 많은 열은 분석에서 제외하거나 적절한 방법으로 대체해야 함


In [1]:
# ! pip install seaborn  # 필요시 주석 해제하여 설치
import seaborn as sns
import pandas as pd

# Seaborn의 예제 데이터셋 'titanic' 불러오기
df = sns.load_dataset('titanic')

# 데이터프레임 요약 정보 확인
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


- `value_counts()` 함수를 사용하면 특정 열의 값 분포를 확인할 수 있음
  - `dropna=False` 옵션을 사용하면 누락 데이터까지 포함한 개수를 확인할 수 있음
  - 누락 데이터의 수를 직접 확인하여 결측 비율이 높은 열을 정제할지 여부를 판단할 수 있음


In [2]:
# 'deck' 열의 값 분포 확인 (결측값 포함)
df['deck'].value_counts(dropna=False)


deck
NaN    688
C       59
B       47
D       33
E       32
A       15
F       13
G        4
Name: count, dtype: int64

- 누락 데이터를 직접 탐색하는 대표적인 방법으로 `isnull()` 함수와 `notnull()` 함수가 있음
  - `isnull()` 함수: 누락 데이터이면 `True`, 유효한 데이터이면 `False`를 반환함
  - `notnull()` 함수: 유효한 데이터이면 `True`, 누락 데이터이면 `False`를 반환함


In [3]:
# 각 열의 누락 여부 확인
df.isnull().head()


   survived  pclass    sex    age  ...   deck  embark_town  alive  alone
0     False   False  False  False  ...   True        False  False  False
1     False   False  False  False  ...  False        False  False  False
2     False   False  False  False  ...   True        False  False  False
3     False   False  False  False  ...  False        False  False  False
4     False   False  False  False  ...   True        False  False  False

[5 rows x 15 columns]

- 누락 데이터의 개수를 계산할 때는 `isnull()` 또는 `notnull()` 함수와 `sum()` 함수를 함께 사용함
  - `sum()` 함수를 적용하면 열별로 `True(1)` 값의 총합을 계산하여 결측값의 개수를 계산할 수 있음


In [4]:
# 각 열의 결측값 개수 계산
df.isnull().sum()


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

### 1-3. 누락 데이터 제거

- 누락 데이터가 포함된 행 또는 열을 제거할 때는 `dropna()` 함수를 사용함
  - `thresh` 옵션을 사용하면 지정된 기준 이상으로 유효한 값(Non-Null)이 존재하는 경우에만 해당 열(또는 행)을 유지함


In [5]:
# 원본 데이터의 열 목록 확인
print(df.columns)


Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='str')


In [6]:
# 결측값이 많은 열 제거 (NaN 값이 500개 이상인 열 제거)
df_thresh = df.dropna(axis=1, thresh=500)

# 제거 후 남은 열 목록 확인
print(df_thresh.columns)


Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alive',
       'alone'],
      dtype='str')


- `dropna()` 함수에 `subset` 인자를 지정하면 특정 열에 한정하여 해당 열에서 누락 데이터가 포함된 행을 삭제할 수 있음
  - `how='any'`: 지정된 열 중 하나라도 누락 데이터가 포함되면 해당 행을 삭제함(기본값)
  - `how='all'`: 지정된 열의 값이 모두 누락 데이터일 때만 삭제함
  - `axis=0`: 행 기준으로 삭제(기본값)


In [7]:
# 'age' 열에서 NaN 값을 가진 행 제거
df_age = df.dropna(subset=['age'], how='any', axis=0)

# 제거 후 남은 행 개수 확인
print(len(df_age))


714


### 1-4. 누락 데이터 대체

- 데이터의 규모가 크고 누락 데이터의 비율이 낮은 경우에는 누락 데이터를 제거하더라도 분석 결과에 큰 영향을 주지 않음
- 그러나 데이터의 크기가 작거나 누락 데이터의 비율이 높은 경우에는 단순 삭제 시 결과가 왜곡되는 문제가 발생함
- 누락 데이터를 제거하지 않고 다른 값으로 대체하여 채워 넣는 방법을 누락 데이터 대체법(Imputation)이라 함
  - 누락 데이터를 적절한 값으로 대체하면 데이터 손실을 줄이고 분석의 왜곡 문제를 완화할 수 있음
  - 평균, 중앙값, 최빈값 등의 대표값으로 대체하는 방법이 일반적으로 사용되며, 보다 정교한 방법으로는 통계적 추정이나 모델 기반 예측(Regression, KNN 등)을 이용할 수 있음


- 앞의 예시처럼 승객의 나이(age)가 누락된 행을 삭제하지 않고 나머지 승객들의 평균 나이로 대체할 수 있음
  - `fillna()` 함수는 누락 데이터를 특정 값으로 대체할 때 사용하며, 기존 데이터프레임에 직접 적용하려면 `inplace=True` 옵션을 추가해야 함
  - 대체값으로 평균, 중앙값, 최빈값 등 대표값(대표 통계량)을 지정할 수 있음


In [8]:
# age 열의 평균값 계산
mean_age = df['age'].mean()
print("평균 나이:", mean_age)


평균 나이: 29.69911764705882


In [9]:
# fillna()를 이용하여 결측값을 평균으로 대체
# df['age'].fillna(mean_age, inplace=True)  # ChainedAssignmentError 발생
df['age'] = df['age'].fillna(mean_age)


In [10]:
# 대체 후 결측값이 모두 제거되었는지 확인
print(df['age'].isnull().sum())


0


- 특정 열에서 가장 자주 등장하는 값(최빈값)으로 누락 데이터를 대체할 수 있음
  - 이 방법은 범주형(categorical) 데이터의 결측값을 처리할 때 자주 사용됨
  - 예를 들어, 승객의 승선 도시(`embark_town`)가 결측된 경우에는 가장 많은 승객이 탑승한 도시명으로 대체할 수 있음


In [11]:
# 일부 데이터 확인
print(df['embark_town'][825:830])


825     Queenstown
826    Southampton
827      Cherbourg
828     Queenstown
829            NaN
Name: embark_town, dtype: str


In [12]:
# 가장 많이 등장하는 값(최빈값) 찾기
most_freq = df['embark_town'].value_counts(dropna=True).idxmax()
print("최빈값:", most_freq)


최빈값: Southampton


In [13]:
# fillna()로 결측값을 최빈값으로 대체
df['embark_town'] = df['embark_town'].fillna(most_freq)


In [14]:
# 대체 결과 확인
print(df['embark_town'][825:830])


825     Queenstown
826    Southampton
827      Cherbourg
828     Queenstown
829    Southampton
Name: embark_town, dtype: str


- 데이터의 특성상 서로 인접한 값들이 유사할 가능성이 높은 경우에는 앞이나 뒤의 값을 이용해 누락 데이터를 보정할 수 있음
  - `method='ffill'` 옵션: 누락 데이터를 바로 이전 행의 값으로 대체함
  - `method='bfill'` 옵션: 누락 데이터를 바로 다음 행의 값으로 대체함
  - 데이터가 연속적이거나 시계열일 때 유용하며 데이터 간의 시간적 또는 순차적 연관성이 있다고 가정할 수 있을 때 적절함


In [15]:
# 타이타닉 데이터 불러오기
df = sns.load_dataset('titanic')

# 일부 데이터 확인
print(df['embark_town'][825:830])


825     Queenstown
826    Southampton
827      Cherbourg
828     Queenstown
829            NaN
Name: embark_town, dtype: str


In [16]:
# 바로 앞의 값으로 결측값 채우기 (ffill)
# df['embark_town'] = df['embark_town'].fillna(method='ffill')  # TypeError 발생
df['embark_town'] = df['embark_town'].ffill()

# 결과 확인
print(df['embark_town'][825:830])


825     Queenstown
826    Southampton
827      Cherbourg
828     Queenstown
829     Queenstown
Name: embark_town, dtype: str


## 2. 범주형 데이터


### 2-1. 범주형 데이터 개요

- 연속형 데이터를 그대로 사용하는 대신 일정한 구간(bin)으로 나누어 분석하면 효율적인 경우가 있음
  - 연속 변수를 일정한 구간으로 나누어 범주형 이산 변수로 변환하는 과정을 구간 분할(binning)이라 함
  - pandas의 `cut()` 함수를 사용하면 연속 데이터를 여러 구간으로 나누어 범주형 데이터로 변환할 수 있음


### 2-2. 범주형 데이터 변환

- pandas의 `cut()` 함수는 연속형 데이터를 여러 구간(bin)으로 나누어 범주형 데이터로 변환함
  - `bins` 옵션: 구간의 경계값 리스트를 지정함
  - `labels` 옵션: 각 구간의 이름을 지정함
  - `include_lowest=True`: 첫 번째 구간에 최솟값을 포함함


In [17]:
import pandas as pd
import numpy as np
import seaborn as sns

# 타이타닉 데이터셋 불러오기
titanic = sns.load_dataset("titanic")

# 분석에 필요한 변수만 선택
df = titanic.loc[:, ["age", "sex", "class", "fare", "survived"]]

# 운임(fare) 변수를 3개 구간으로 나누기 위한 경계값 계산
count, bin_dividers = np.histogram(df["fare"], bins=3)
print(bin_dividers)


[  0.     170.7764 341.5528 512.3292]


In [18]:
# 각 구간의 이름 정의
bin_names = ["Low", "Medium", "High"]

# cut()을 이용해 구간화된 범주형 데이터 생성
df["fare_bin"] = pd.cut(
    x=df["fare"],             # 구간화할 변수
    bins=bin_dividers,         # 구간 경계값
    labels=bin_names,          # 구간 이름
    include_lowest=True        # 최솟값 포함 여부
)

# 결과 확인
print(df[["fare", "fare_bin"]].head())


      fare fare_bin
0   7.2500      Low
1  71.2833      Low
2   7.9250      Low
3  53.1000      Low
4   8.0500      Low
